# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### UV Setup

In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [ ]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

/bin/bash: line 1: ./.venv/bin/activate: No such file or directory


### Colab Setup
Connect to Google Drive so you can load the dataset and save your results.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip uninstall -y torch torchvision torchaudio transformers protobuf tensorflow tensorflow-cpu
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q vllm==0.8.5 transformers==4.51.3 accelerate bitsandbytes tqdm sympy antlr4-python3-runtime==4.11.1
!pip install -q protobuf==4.25.3

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 113.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [63]:
import torch
torch.cuda.get_device_name(0)

'NVIDIA A100-SXM4-40GB'

In [7]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/results/voting_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

INFO 05-30 19:15:55 [__init__.py:239] Automatically detected platform cuda.


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [64]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [65]:
SYSTEM_PROMPT_MATH = (
    "Solve the math problem. Show only the necessary reasoning. "
    "End with the final answer inside \\boxed{}. "
    "If there are multiple answers, put them comma-separated inside one box, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "Solve the multiple-choice math problem. "
    "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [66]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

print("Model loaded.")

INFO 05-30 20:50:48 [config.py:717] This model supports multiple tasks: {'generate', 'embed', 'classify', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 05-30 20:50:48 [config.py:830] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-30 20:50:48 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-30 20:52:23 [core_client.py:439] Core engine process 0 ready.
Model loaded.


In [67]:
NUM_VOTES = 3

In [68]:
sampling_params = SamplingParams(
    n=NUM_VOTES,
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [69]:
# Build prompts for first N entries
N = 100
prompts = []
for item in data[:N]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating {NUM_VOTES} responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)


Generating 3 responses for 100 questions...


Processed prompts:   0%|          | 0/300 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [70]:
responses = [
    [choice.text.strip() for choice in out.outputs]
    for out in outputs
]

for i, votes in enumerate(responses):
    # print(f"\n── Preview Question {i} (id={preview_data[i].get('id')}) ──")
    for j, txt in enumerate(votes, start=1):
        print(f"\nVote {j}:")
        print(txt[:500], "..." if len(txt) > 500 else "")


Vote 1:
Okay, let's see. The problem is to find the sum of the first 325 positive even whole numbers. Hmm, first, I need to remember what the first few positive even whole numbers are. Let me list them out to get a sense. The first positive even whole number is 2, then 4, 6, 8, and so on. So they're 2, 4, 6, 8,..., up to the 325th term. 

I think this is an arithmetic sequence. Let me confirm. The nth term of an arithmetic sequence is given by a_n = a_1 + (n - 1)d, where a_1 is the first term, d is the  ...

Vote 2:
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even numbers are. Positive even whole numbers start from 2, right? So the first one is 2, the second is 4, third is 6, and so on. So the nth positive even whole number is 2n. Let me confirm that. For n=1, 2*1=2, which is correct. n=2, 2*2=4, correct. Yeah, that seems right.

So the problem is asking for the sum of the first 325 terms 

### Generate with Transformers (for Datahub)

In [ ]:
# import torch

# print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))
# else:
#     print("No GPU detected")

In [ ]:
# # Build prompts for first 5 entries
# MAX_TOKENS = 2048
# N = 10 # Number of questions

# prompts = []
# for item in data[:N]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=2048 #16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")



In [ ]:
print(prompts[0])
print(responses[0])

In [ ]:
# The whole dataset in batch sizes of 5

MAX_TOKENS = 2048
BATCH_SIZE = 2

responses = []

print(f"Generating responses for {len(data)} questions in batches of {BATCH_SIZE}...")

for start in range(0, len(data), BATCH_SIZE):

    batch = data[start:start+BATCH_SIZE]

    # Build prompts
    prompts = []
    for item in batch:
        system, user = build_prompt(item["question"], item.get("options"))

        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )

        prompts.append(prompt_text)

    # Tokenize batch
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    # Generate
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode
    for i, out in enumerate(output_ids):
        new_tokens = out[inputs["input_ids"].shape[1]:]

        text = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        responses.append(text)

    # Free GPU memory between batches
    del inputs, output_ids
    torch.cuda.empty_cache()

    print(f"Completed {min(start+BATCH_SIZE, len(data))}/{len(data)}")

# Preview first few outputs
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [71]:
from collections import Counter
import pandas as pd

# ---------- Answer extraction helpers ----------

def extract_boxed(text: str) -> str:
    """Extract the last \boxed{...} content if present. Handles simple boxed answers."""
    if not text:
        return ""
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    return matches[-1].strip() if matches else ""


def extract_final_answer_line(text: str) -> str:
    """Extract answer after a Final Answer marker if present."""
    if not text:
        return ""
    patterns = [
        r"final\s+answer\s*[:=]\s*(.+)",
        r"answer\s*[:=]\s*(.+)",
    ]
    for pat in patterns:
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            ans = matches[-1].strip().split("\n")[0].strip()
            return ans
    return ""


def normalize_answer(ans: str) -> str:
    """Light normalization for comparing/voting extracted answers."""
    if ans is None:
        return ""
    ans = str(ans).strip()
    ans = re.sub(r"^\\boxed\{(.+)\}$", r"\1", ans).strip()
    ans = ans.replace("$", "").strip()
    ans = re.sub(r"^\((.*)\)$", r"\1", ans).strip()
    ans = re.sub(r"\s+", " ", ans)
    ans = ans.rstrip(".。,")
    return ans


def extract_answer(text: str, is_mcq: bool = False) -> str:
    """Extract a clean final answer from a model response."""
    boxed = normalize_answer(extract_boxed(text))
    if boxed:
        ans = boxed
    else:
        ans = normalize_answer(extract_final_answer_line(text))

    if is_mcq:
        # Prefer a single MCQ letter from the extracted answer.
        m = re.search(r"\b([A-Z])\b", ans.upper())
        if m:
            return m.group(1)
        # Fallback: last standalone capital letter in the whole response.
        matches = re.findall(r"\b([A-Z])\b", text.upper())
        return matches[-1] if matches else ""

    return ans


def majority_vote(answers: list[str]) -> str:
    """Return majority answer after normalization; if no majority, return first parseable answer."""
    cleaned = [normalize_answer(a) for a in answers if normalize_answer(a)]
    if not cleaned:
        return ""
    counts = Counter(cleaned)
    top_answer, top_count = counts.most_common(1)[0]
    if top_count >= 2:
        return top_answer
    return cleaned[0]

In [72]:
def score_mcq_answer(pred_answer: str, gold_letter: str) -> bool:
    return normalize_answer(pred_answer).upper() == str(gold_letter).strip().upper()

# Load Judger for free-form scoring
sys.path.insert(0, "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition")
from judger import Judger
judger = Judger(strict_extract=False)

def score_free_answer(pred_answer: str, gold) -> bool:
    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return bool(judger.auto_judge(
            pred=pred_answer,
            gold=gold_list,
            options=[[]] * len(gold_list),
        ))
    except Exception:
        return False


def score_answer(pred_answer: str, gold, is_mcq: bool) -> bool:
    if is_mcq:
        return score_mcq_answer(pred_answer, gold)
    return score_free_answer(pred_answer, gold)

def boxed_response(ans):
    ans = "" if ans is None else str(ans).strip()
    if ans == "":
        return ""
    return f"\\boxed{{{ans}}}"

In [73]:
results = []
rows = []

# --- Official-judger scoring helpers ---

def boxed_response(ans):
    """
    Wrap an extracted answer so the official Judger can extract it reliably.
    Example: "5/8" -> "\\boxed{5/8}"
    """
    ans = "" if ans is None else str(ans).strip()
    if ans == "":
        return ""
    return f"\\boxed{{{ans}}}"


def get_item_field(item, possible_keys, default=None):
    """
    Try several possible field names because datasets sometimes use slightly
    different column/key names.
    """
    for key in possible_keys:
        if key in item and item.get(key) is not None:
            return item.get(key)
    return default


def score_with_official_judger(ans, item):
    """
    Score an extracted answer using the official course Judger.

    The Judger expects the full model response, not just a bare answer,
    so we wrap the answer in \\boxed{...}.
    """
    ans = "" if ans is None else str(ans).strip()
    if ans == "":
        return False

    gold = get_item_field(item, ["answer", "gold"], default=[])
    type_sequence = get_item_field(item, ["type_sequence", "type", "answer_type"], default=None)
    options = get_item_field(item, ["options"], default=[])
    precision = get_item_field(item, ["precision"], default=1e-8)

    # The official judger expects lists for gold/type_sequence/options.
    # Gold is often already a list like ["5/8"], but this protects us.
    if not isinstance(gold, list):
        gold = [gold]

    if type_sequence is None:
        # Fallback: infer MCQ vs non-MCQ if type_sequence is missing.
        # Prefer official field when available.
        is_mcq = bool(options)
        type_sequence = ["MCS"] if is_mcq else ["NV"]

    if not isinstance(type_sequence, list):
        type_sequence = [type_sequence]

    # options should be a list with one entry per answer slot.
    # For MCQ it may be ["A","B","C",...] instead of [["A","B","C",...]],
    # so wrap if needed.
    if options is None:
        options = []

    if len(type_sequence) == 1:
        if options == []:
            options_for_judge = [[]]
        elif isinstance(options, list) and all(isinstance(x, str) for x in options):
            options_for_judge = [options]
        else:
            options_for_judge = options
    else:
        options_for_judge = options
        if not isinstance(options_for_judge, list) or len(options_for_judge) != len(type_sequence):
            options_for_judge = [[] for _ in type_sequence]

    try:
        return bool(
            judger.judge(
                pred=boxed_response(ans),
                gold=gold,
                type_sequence=type_sequence,
                options=options_for_judge,
                precision=precision,
            )
        )
    except Exception as e:
        # Useful while debugging, but do not crash the whole scoring run.
        return False


# --- Main voting/scoring loop ---

active_data = data[:N] if N is not None else data

for item, vote_responses in tqdm(
    zip(active_data, responses),
    total=len(active_data),
    desc="Scoring votes"
):
    is_mcq = bool(item.get("options"))
    gold = item.get("answer", "")

    vote_answers = [extract_answer(resp, is_mcq=is_mcq) for resp in vote_responses]
    chosen_answer = majority_vote(vote_answers)

    # Use official Judger, not simple string comparison.
    chosen_correct = score_with_official_judger(chosen_answer, item)
    vote_correct = [
        score_with_official_judger(ans, item) if ans else False
        for ans in vote_answers
    ]

    # Ensure exactly 3 display columns even if NUM_VOTES changes later.
    padded_answers = (vote_answers + [""] * 3)[:3]
    padded_correct = (vote_correct + [False] * 3)[:3]
    padded_responses = (vote_responses + [""] * 3)[:3]

    row = {
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "question": item.get("question", "")[:250],
        "gold": gold,
        "type_sequence": item.get("type_sequence"),
        "options": item.get("options"),
        "answer_1": padded_answers[0],
        "answer_2": padded_answers[1],
        "answer_3": padded_answers[2],
        "answer_1_correct": padded_correct[0],
        "answer_2_correct": padded_correct[1],
        "answer_3_correct": padded_correct[2],
        "majority_answer": chosen_answer,
        "majority_correct": chosen_correct,
        "majority_boxed_response": boxed_response(chosen_answer),
        "response_1": padded_responses[0],
        "response_2": padded_responses[1],
        "response_3": padded_responses[2],
    }
    rows.append(row)

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "response": boxed_response(chosen_answer),  # boxed response for official-style judging
        "raw_answer": chosen_answer,                # bare extracted answer
        "raw_responses": vote_responses,            # all raw generations
        "vote_answers": vote_answers,               # extracted answers from each generation
        "correct": chosen_correct,
    })

vote_df = pd.DataFrame(rows)

print(f"Scoring complete. {len(results)} results.")
print("\nPreview of voting dataframe:")
display(vote_df[[
    "id", "is_mcq", "gold",
    "answer_1", "answer_2", "answer_3",
    "majority_answer", "majority_boxed_response", "majority_correct"
]].head(20))

Scoring votes:   0%|          | 0/100 [00:00<?, ?it/s]

['143.224229233795', '2.32624773420025']
['143.224229233795', '2.32624773420025']
['143.224229233795', '2.32624773420025']
['143.224229233795', '2.32624773420025']


Scoring votes:   3%|▎         | 3/100 [00:00<00:06, 15.46it/s]

['62.7777777777778', '335.927777777778', '604.67']
['62.7777777777778', '335.927777777778', '604.67']


Scoring votes:   6%|▌         | 6/100 [00:00<00:05, 17.26it/s]

...

Scoring votes:  81%|████████  | 81/100 [00:02<00:00, 35.37it/s]

['13^2 + (x-4)^2 = x^2', '23.125']
['13^2 + (x-4)^2 = x^2', '23.125']
['13^2 + (x-4)^2 = x^2', '23.125']
['sqrt(51^2+56^2)', 'N', '65.3246', 'E']
['sqrt(51^2+56^2)', 'N', '65.3246', 'E']
['sqrt(51^2+56^2)', 'N', '65.3246', 'E']
['sqrt(51^2+56^2)', 'N', '65.3246', 'E']
['yes', 'yes', 'no']
['yes', 'yes', 'no']
['yes', 'yes', 'no']
['yes', 'yes', 'no']


Scoring votes: 100%|██████████| 100/100 [00:02<00:00, 35.37it/s]

Scoring complete. 100 results.

Preview of voting dataframe:


,id,is_mcq,gold,answer_1,answer_2,answer_3,majority_answer,majority_boxed_response,majority_correct
0,0,False,[325*(1+325)],105950,105950,105950,105950,\boxed{105950},True
1,1,True,F,E,B,B,B,\boxed{B},False
2,2,False,"[143.224229233795, 2.32624773420025]",2.33,2.33,2.33,2.33,\boxed{2.33},False
3,3,False,[5/8],5/8,,,5/8,\boxed{5/8},True
4,4,True,C,C,C,C,C,\boxed{C},True
5,5,False,"[62.7777777777778, 335.927777777778, 604.67]","62.78, 335.93, 604.67","62.78, 335.93, 604.67","62.78, 335.93, 604.67","62.78, 335.93, 604.67","\boxed{62.78, 335.93, 604.67}",False
6,6,False,"[G, B]","G, B","G, B","G, B","G, B","\boxed{G, B}",True
7,7,False,[1.44444444444444],,,,,,False
8,8,False,[(1/2)^[(1999-1963)/31]],0.447,0.447,0.447,0.447,\boxed{0.447},False
9,9,True,A,A,A,A,A,\boxed{A},True


In [77]:
vote_df[vote_df["is_mcq"] == False].get(["gold", "answer_1", "answer_2", "answer_3", "majority_answer", "majority_correct"]).head(20)


,gold,answer_1,answer_2,answer_3,majority_answer,majority_correct
0,[325*(1+325)],105950,105950,105950,105950,True
2,"[143.224229233795, 2.32624773420025]",2.33,2.33,2.33,2.33,False
3,[5/8],5/8,,,5/8,True
5,"[62.7777777777778, 335.927777777778, 604.67]","62.78, 335.93, 604.67","62.78, 335.93, 604.67","62.78, 335.93, 604.67","62.78, 335.93, 604.67",False
6,"[G, B]","G, B","G, B","G, B","G, B",True
7,[1.44444444444444],,,,,False
8,[(1/2)^[(1999-1963)/31]],0.447,0.447,0.447,0.447,False
12,"[380, 315, 13, 310]","380, 315, 14, 310",310,"380, 315, 14, 310","380, 315, 14, 310",True
15,"[B, C, A]","B, C, A","B, C, A","B, C, A","B, C, A",True
16,"[atan(4.76), pi]","1.364, 3.142","1.363, 3.142","1.364, 3.142","1.364, 3.142",False


## 8. Summary

Print accuracy broken down by question type.

In [75]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   28 /   38  (73.68%)
  Free-form  :   28 /   62  (45.16%)
  Overall    :   56 /  100  (56.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [38]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 10 records to /content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!